In [1]:
import pandas as pd

In [2]:
filepath = '../data/데이터_rev.01_2025.04.04_산학용.xlsx'

In [3]:
df = pd.read_excel(filepath, sheet_name=None, skiprows=[1])
sheet_name_list = list(df.keys())

In [4]:
class WorkUnit:
    def __init__(self, unit_id, x, y, dx, dy):
        self.unit_id = unit_id
        self.x = int(x * 10)
        self.y = int(y * 10)
        self.dx = int(dx * 10)
        self.dy = int(dy * 10)


class WorkArea:
    def __init__(self, group_id, surface_id_list, priority, lug_condition, indoor_outdoor_condition, L_limit_of_block,
                 B_limit_of_block, H_limit_of_block, W_limit_of_block, TP_condition, TP_direction, L, B):
        self.group_id = group_id
        self.surface_id_list = surface_id_list
        self.priority = priority
        self.indoor_outdoor_condition = indoor_outdoor_condition
        self.lug_condition = lug_condition
        self.L_limit_of_block = int(L_limit_of_block * 10)
        self.B_limit_of_block = int(B_limit_of_block * 10)
        self.H_limit_of_block = int(H_limit_of_block * 10)
        self.W_limit_of_block = int(W_limit_of_block * 10)
        self.TP_condition = TP_condition
        # [방향1, 방향2, 방향3, 방향4]에 대한 boolean
        self.TP_direction = TP_direction
        self.L = int(L * 10)
        self.B = int(B * 10)
        self.unavailable_area_x = None
        self.unavailable_area_y = None
        self.unavailable_area_L = None
        self.unavailable_area_B = None
        self.crane_operation_dict = dict()
        self.work_unit_dict = dict()

    def add_unavailable_area(self, x, y, L, B):
        self.unavailable_area_x = int(x * 10)
        self.unavailable_area_y = int(y * 10)
        self.unavailable_area_L = int(L * 10)
        self.unavailable_area_B = int(B * 10)


class Crane:
    def __init__(self, Crane_id, condition):
        self.Crane_id = Crane_id
        self.condition = condition
        self.unavailable_time_list = list()


class Block:
    def __init__(self, ship_type, project_number, block_number,
                 allocation_start_date, allocation_end_date, processing_time, TO_date, PE_date,
                 length, breadth, height, weight, indoor_outdoor_condition, lug_direction, allocate_condtion):
        self.ship_type = ship_type
        self.project_number = project_number
        self.block_number = block_number
        self.allocation_start_date = allocation_start_date
        self.allocation_end_date = allocation_end_date
        self.processing_time = processing_time
        self.TO_date = TO_date
        self.PE_date = PE_date
        self.length = int(length * 10)
        self.breadth = int(breadth * 10)
        self.height = int(height * 10)
        self.weight = int(weight * 10)
        self.indoor_outdoor_condition = indoor_outdoor_condition
        self.lug_direction = lug_direction
        self.allocate_condtion = allocate_condtion
        self.group_id = None
        self.surface_id_list = None
        self.x_location = None
        self.y_location = None

    def adjust_time(self, calendar):
        pass

    def get_location(self, group_id, x_location, y_location):
        # 향후 배치 확정 블록 데이터 존재 시 좌표를 정반 그룹과 위치에 맞춰 변환하는 코드 추가 구현
        self.surface_id_list = None
        pass




In [24]:
df['UNAL_WORKDAY']

,달력일자,휴일여부
0,2019-01-01,1
1,2019-01-02,0
2,2019-01-03,0
3,2019-01-04,0
4,2019-01-05,1
...,...,...
361,2019-12-28,1
362,2019-12-29,1
363,2019-12-30,0
364,2019-12-31,0


In [21]:
df['BLK'].loc[0, '착수일']

Timestamp('2019-04-03 00:00:00')

In [23]:
df['BLK'].loc[0,'착수일'] in df['UNAL_WORKDAY']['달력일자'].values

True

In [49]:
df_block = df['BLK']
df_workday = df['UNAL_WORKDAY']

In [79]:
config = dict()
config['data_start_date'] = '2019-06-04'
config['data_duration'] = 14
config['only_workingday'] = True
config['time_limit'] = 60

In [80]:
df_calendar = df['UNAL_WORKDAY']
start_date = pd.to_datetime(config['data_start_date'])
while True:
    row = df_calendar[df_calendar['달력일자'] == start_date]
    if row.empty or row.iloc[0]['휴일여부'] == 0:
        break
    start_date += pd.Timedelta(days=1)
df_after_start = df_calendar[df_calendar['달력일자'] >= start_date].copy()
if config['only_workingday']:
    df_after_start = df_after_start[df_after_start['휴일여부'] == 0]
# end_date 계산
if len(df_after_start) < config['data_duration']:
    raise ValueError("달력 데이터가 부족합니다.")
block_end_date = df_after_start.iloc[config['data_duration'] - 1]['달력일자']
while True:
    row = df_calendar[df_calendar['달력일자'] == block_end_date]
    if row.empty or row.iloc[0]['휴일여부'] == 0:
        break
    block_end_date += pd.Timedelta(days=1)

In [81]:
start_date, block_end_date

(Timestamp('2019-06-04 00:00:00'), Timestamp('2019-06-24 00:00:00'))

In [88]:
df_block = df['BLK']
df_block[['착수일', '완료일', 'TO일정', 'PE일정']] \
    = df_block[['착수일', '완료일', 'TO일정', 'PE일정']].apply(pd.to_datetime, errors='coerce')
df_block_filtered = df_block[(df_block['착수일'] >= start_date) & (df_block['착수일'] <= block_end_date)]
end_date = df_block_filtered['PE일정'].max()

In [89]:
end_date

Timestamp('2019-07-24 00:00:00')

In [90]:
# self.end_date부터 평일 기준 3일 뒤 날짜로 업데이트
df_after_pe = df_calendar[df_calendar['달력일자'] >= end_date]
df_after_pe = df_after_pe[df_after_pe['휴일여부'] == 0]

if len(df_after_pe) < 4:
    raise ValueError("PE일정 이후 평일이 부족하여 end_date 계산 불가")

end_date = df_after_pe.iloc[3]['달력일자']

In [91]:
end_date

Timestamp('2019-07-29 00:00:00')

In [95]:
calendar1 = dict()
calendar2 = dict()
date = start_date
idx = 0

while True:
    # 평일이면 기록
    if df_calendar.loc[df_calendar['달력일자'] == date, '휴일여부'].iloc[0] == 0:
        calendar1[date] = idx
        calendar2[idx] = date
        idx += 1
    # end_date면 기록 후 종료
    if date == end_date:
        break
    date += pd.Timedelta(days=1)


In [98]:
calendar1.keys()

dict_keys([Timestamp('2019-06-04 00:00:00'), Timestamp('2019-06-05 00:00:00'), Timestamp('2019-06-07 00:00:00'), Timestamp('2019-06-10 00:00:00'), Timestamp('2019-06-11 00:00:00'), Timestamp('2019-06-12 00:00:00'), Timestamp('2019-06-13 00:00:00'), Timestamp('2019-06-14 00:00:00'), Timestamp('2019-06-17 00:00:00'), Timestamp('2019-06-18 00:00:00'), Timestamp('2019-06-19 00:00:00'), Timestamp('2019-06-20 00:00:00'), Timestamp('2019-06-21 00:00:00'), Timestamp('2019-06-24 00:00:00'), Timestamp('2019-06-25 00:00:00'), Timestamp('2019-06-26 00:00:00'), Timestamp('2019-06-27 00:00:00'), Timestamp('2019-06-28 00:00:00'), Timestamp('2019-07-01 00:00:00'), Timestamp('2019-07-02 00:00:00'), Timestamp('2019-07-03 00:00:00'), Timestamp('2019-07-04 00:00:00'), Timestamp('2019-07-05 00:00:00'), Timestamp('2019-07-08 00:00:00'), Timestamp('2019-07-09 00:00:00'), Timestamp('2019-07-10 00:00:00'), Timestamp('2019-07-11 00:00:00'), Timestamp('2019-07-12 00:00:00'), Timestamp('2019-07-15 00:00:00'), Tim

In [97]:
calendar2

{0: Timestamp('2019-06-04 00:00:00'),
 1: Timestamp('2019-06-05 00:00:00'),
 2: Timestamp('2019-06-07 00:00:00'),
 3: Timestamp('2019-06-10 00:00:00'),
 4: Timestamp('2019-06-11 00:00:00'),
 5: Timestamp('2019-06-12 00:00:00'),
 6: Timestamp('2019-06-13 00:00:00'),
 7: Timestamp('2019-06-14 00:00:00'),
 8: Timestamp('2019-06-17 00:00:00'),
 9: Timestamp('2019-06-18 00:00:00'),
 10: Timestamp('2019-06-19 00:00:00'),
 11: Timestamp('2019-06-20 00:00:00'),
 12: Timestamp('2019-06-21 00:00:00'),
 13: Timestamp('2019-06-24 00:00:00'),
 14: Timestamp('2019-06-25 00:00:00'),
 15: Timestamp('2019-06-26 00:00:00'),
 16: Timestamp('2019-06-27 00:00:00'),
 17: Timestamp('2019-06-28 00:00:00'),
 18: Timestamp('2019-07-01 00:00:00'),
 19: Timestamp('2019-07-02 00:00:00'),
 20: Timestamp('2019-07-03 00:00:00'),
 21: Timestamp('2019-07-04 00:00:00'),
 22: Timestamp('2019-07-05 00:00:00'),
 23: Timestamp('2019-07-08 00:00:00'),
 24: Timestamp('2019-07-09 00:00:00'),
 25: Timestamp('2019-07-10 00:00:00

In [50]:
# 필요한 날짜 열
date_cols = ['착수일', '완료일', 'TO일정', 'PE일정']

# 모든 날짜들을 하나의 Series로 합치기
all_dates = pd.concat([df_block[col] for col in date_cols])

# 중복 제거 및 NaT 제거
unique_dates = all_dates.dropna().unique()

# 달력 데이터에서 휴일 여부 확인
is_holiday = df_workday[df_workday['달력일자'].isin(unique_dates)]['휴일여부']

# 모두 평일(휴일여부 == 0)인지 확인
all_workdays = (is_holiday == 0).all()

print("모든 날짜가 평일인가요?", all_workdays)

모든 날짜가 평일인가요? False


In [52]:
# 휴일인 날짜 set 만들기
holiday_set = set(df_workday[df_workday['휴일여부'] == 1]['달력일자'])

# 결과 저장할 DataFrame
problem_rows = []

for idx, row in df_block.iterrows():
    problem = {}
    for col in ['착수일', '완료일', 'TO일정', 'PE일정']:
        date = row[col]
        if pd.notna(date) and date in holiday_set:
            problem[col] = True
        else:
            problem[col] = False
    if any(problem.values()):
        problem['index'] = idx
        problem_rows.append(problem)

# DataFrame으로 변환
df_problem = pd.DataFrame(problem_rows).set_index('index')
df_problem


,착수일,완료일,TO일정,PE일정
index,,,,
6,False,False,False,True
7,False,False,True,False
9,False,True,False,False
12,False,True,False,False
18,False,False,True,False
...,...,...,...,...
187,False,True,True,False
188,False,True,True,False
192,False,True,False,False
